In [1]:
!pip install pytorch-crf -q

In [2]:
from torchcrf import CRF

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from pathlib import Path
import random
import numpy as np

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [5]:
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [6]:
datasets = {
    "Barishal": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Barishal_NER.csv"),
    "Chittagong": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Chittagong_NER.csv"),
    "Mymensingh": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Mymensingh_NER.csv"),
    "Noakhali": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Noakhali_NER.csv"),
    "Sylhet": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Sylhet_NER.csv")
}

print("Datasets loaded.")

Datasets loaded.


In [7]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [8]:
sentence_data = {}

for dialect, df in datasets.items():

    df["Sentence #"] = df["Sentence #"].ffill()

    word_column = [
        c for c in df.columns
        if c.endswith("_word")
    ][0]

    sentences = []

    for sentence_id, group in df.groupby("Sentence #"):

        sentences.append({
            "sentence_id": int(
                str(sentence_id).replace("Sentence: ", "")
            ),
            "tokens": group[word_column].astype(str).tolist(),
            "tags": group["bio_tag"].astype(str).tolist()
        })

    sentence_data[dialect] = sentences

print("Sentence data created.")

for dialect in sentence_data:
    print(dialect, len(sentence_data[dialect]))

Sentence data created.
Barishal 3481
Chittagong 3481
Mymensingh 3481
Noakhali 3481
Sylhet 3481


In [9]:
import json

with open(
    "/kaggle/input/datasets/arnav1605/ancholink-ner/within_dialect_splits.json",
    "r"
) as f:
    splits = json.load(f)

print("Splits loaded.")

Splits loaded.


In [10]:
tags = set()

for sentences in sentence_data.values():
    for sentence in sentences:
        tags.update(sentence["tags"])

tags = sorted(tags)

tag_to_id = {
    tag: i for i, tag in enumerate(tags)
}

id_to_tag = {
    i: tag for tag, i in tag_to_id.items()
}

print("Number of tags:", len(tags))
print(tag_to_id)

Number of tags: 20
{'\nI-LOC': 0, 'B-ANI': 1, 'B-COL': 2, 'B-FOOD': 3, 'B-LOC': 4, 'B-OBJ': 5, 'B-ORG': 6, 'B-PER': 7, 'B-REL': 8, 'B-ROLE': 9, 'I-ANI': 10, 'I-COL': 11, 'I-FOOD': 12, 'I-LOC': 13, 'I-OBJ': 14, 'I-ORG': 15, 'I-PER': 16, 'I-REL': 17, 'I-ROLE': 18, 'O': 19}


In [11]:
def build_vocab(sentences):

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for sentence in sentences:

        for token in sentence["tokens"]:

            if token not in vocab:
                vocab[token] = len(vocab)

    return vocab

In [12]:
class NERDataset(Dataset):

    def __init__(self, sentences, vocab):

        self.data = [
            encode_sentence(sentence, vocab)
            for sentence in sentences
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        tokens, tags = self.data[index]

        return (
            torch.tensor(tokens, dtype=torch.long),
            torch.tensor(tags, dtype=torch.long)
        )

In [13]:
def collate_fn(batch):

    tokens, tags = zip(*batch)

    max_length = max(len(x) for x in tokens)

    padded_tokens = []
    padded_tags = []
    masks = []

    for x, y in zip(tokens, tags):

        length = len(x)
        padding = max_length - length

        padded_tokens.append(
            torch.cat([
                x,
                torch.zeros(padding, dtype=torch.long)
            ])
        )

        padded_tags.append(
            torch.cat([
                y,
                torch.zeros(padding, dtype=torch.long)
            ])
        )

        masks.append(
            torch.tensor(
                [1] * length + [0] * padding,
                dtype=torch.bool
            )
        )

    return (
        torch.stack(padded_tokens),
        torch.stack(padded_tags),
        torch.stack(masks)
    )

In [14]:
vocabularies = {}

for dialect in sentence_data:

    train_ids = set(splits[dialect]["train"])

    train_sentences = [
        s for s in sentence_data[dialect]
        if s["sentence_id"] in train_ids
    ]

    vocabularies[dialect] = build_vocab(train_sentences)

    print(
        dialect,
        "| Vocabulary size:", len(vocabularies[dialect])
    )

Barishal | Vocabulary size: 3371
Chittagong | Vocabulary size: 4317
Mymensingh | Vocabulary size: 3467
Noakhali | Vocabulary size: 4053
Sylhet | Vocabulary size: 3845


In [15]:
def encode_sentence(sentence, vocab):

    token_ids = [
        vocab.get(token, vocab["<UNK>"])
        for token in sentence["tokens"]
    ]

    tag_ids = [
        tag_to_id[tag]
        for tag in sentence["tags"]
    ]

    return token_ids, tag_ids

In [16]:
class BiLSTMCRF(nn.Module):

    def __init__(
        self,
        vocab_size,
        tag_size,
        embedding_dim=100,
        hidden_dim=128,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            tag_size
        )

        self.crf = CRF(
            tag_size,
            batch_first=True
        )

    def forward(self, tokens, tags=None, mask=None):

        x = self.embedding(tokens)

        x, _ = self.lstm(x)

        x = self.dropout(x)

        emissions = self.fc(x)

        if tags is not None:

            loss = -self.crf(
                emissions,
                tags,
                mask=mask,
                reduction="mean"
            )

            return loss

        predictions = self.crf.decode(
            emissions,
            mask=mask
        )

        return predictions

In [17]:
from torch.utils.data import DataLoader

train_loaders = {}
test_loaders = {}

for dialect in sentence_data:

    train_ids = set(splits[dialect]["train"])
    test_ids = set(splits[dialect]["test"])

    train_sentences = [
        s for s in sentence_data[dialect]
        if s["sentence_id"] in train_ids
    ]

    test_sentences = [
        s for s in sentence_data[dialect]
        if s["sentence_id"] in test_ids
    ]

    train_dataset = NERDataset(
        train_sentences,
        vocabularies[dialect]
    )

    test_dataset = NERDataset(
        test_sentences,
        vocabularies[dialect]
    )

    train_loaders[dialect] = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        collate_fn=collate_fn
    )

    test_loaders[dialect] = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=collate_fn
    )

    print(
        dialect,
        "| Train:", len(train_dataset),
        "| Test:", len(test_dataset)
    )

Barishal | Train: 2784 | Test: 697
Chittagong | Train: 2784 | Test: 697
Mymensingh | Train: 2784 | Test: 697
Noakhali | Train: 2784 | Test: 697
Sylhet | Train: 2784 | Test: 697


In [18]:
def train_model(model, train_loader, epochs=10, lr=1e-3):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for tokens, tags, mask in train_loader:

            tokens = tokens.to(device)
            tags = tags.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            loss = model(
                tokens,
                tags=tags,
                mask=mask
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        average_loss = total_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"- Loss: {average_loss:.4f}"
        )

In [19]:
def get_entities(tags):

    entities = []
    start = None
    entity_type = None

    for i, tag in enumerate(tags + ["O"]):

        if tag.startswith("B-"):

            if start is not None:
                entities.append(
                    (start, i - 1, entity_type)
                )

            start = i
            entity_type = tag[2:]

        elif tag.startswith("I-"):

            if start is None:
                start = i
                entity_type = tag[2:]

        else:

            if start is not None:
                entities.append(
                    (start, i - 1, entity_type)
                )

            start = None
            entity_type = None

    return set(entities)

In [20]:
def evaluate_model(model, test_loader):

    model.eval()

    total_correct = 0
    total_predicted = 0
    total_actual = 0

    with torch.no_grad():

        for tokens, tags, mask in test_loader:

            tokens = tokens.to(device)
            mask = mask.to(device)

            predictions = model(
                tokens,
                mask=mask
            )

            for i, pred in enumerate(predictions):

                length = mask[i].sum().item()

                true_tags = [
                    id_to_tag[x.item()]
                    for x in tags[i][:length]
                ]

                pred_tags = [
                    id_to_tag[x]
                    for x in pred
                ]

                true_entities = get_entities(true_tags)
                pred_entities = get_entities(pred_tags)

                total_correct += len(
                    true_entities & pred_entities
                )

                total_actual += len(true_entities)
                total_predicted += len(pred_entities)

    precision = (
        total_correct / total_predicted
        if total_predicted > 0 else 0
    )

    recall = (
        total_correct / total_actual
        if total_actual > 0 else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0
    )

    return precision, recall, f1

In [21]:
bilstm_models = {}
bilstm_results = {}

for dialect in sentence_data:

    print("\n" + "=" * 50)
    print("Training:", dialect)
    print("=" * 50)

    model = BiLSTMCRF(
        vocab_size=len(vocabularies[dialect]),
        tag_size=len(tag_to_id)
    ).to(device)

    train_model(
        model,
        train_loaders[dialect],
        epochs=20,
        lr=1e-3
    )

    precision, recall, f1 = evaluate_model(
        model,
        test_loaders[dialect]
    )

    bilstm_models[dialect] = model

    bilstm_results[dialect] = {
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

    print(
        f"\n{dialect} Results:"
        f"\nPrecision: {precision:.4f}"
        f"\nRecall:    {recall:.4f}"
        f"\nF1:        {f1:.4f}"
    )


Training: Barishal
Epoch 1/20 - Loss: 6.2224
Epoch 2/20 - Loss: 2.8470
Epoch 3/20 - Loss: 2.3073
Epoch 4/20 - Loss: 1.8313
Epoch 5/20 - Loss: 1.4399
Epoch 6/20 - Loss: 1.1297
Epoch 7/20 - Loss: 0.8856
Epoch 8/20 - Loss: 0.6825
Epoch 9/20 - Loss: 0.5216
Epoch 10/20 - Loss: 0.3913
Epoch 11/20 - Loss: 0.2839
Epoch 12/20 - Loss: 0.2156
Epoch 13/20 - Loss: 0.1615
Epoch 14/20 - Loss: 0.1196
Epoch 15/20 - Loss: 0.0994
Epoch 16/20 - Loss: 0.0717
Epoch 17/20 - Loss: 0.0587
Epoch 18/20 - Loss: 0.0495
Epoch 19/20 - Loss: 0.0414
Epoch 20/20 - Loss: 0.0347

Barishal Results:
Precision: 0.8191
Recall:    0.6030
F1:        0.6946

Training: Chittagong
Epoch 1/20 - Loss: 6.2750
Epoch 2/20 - Loss: 3.0074
Epoch 3/20 - Loss: 2.5197
Epoch 4/20 - Loss: 2.0784
Epoch 5/20 - Loss: 1.7149
Epoch 6/20 - Loss: 1.3959
Epoch 7/20 - Loss: 1.1455
Epoch 8/20 - Loss: 0.9057
Epoch 9/20 - Loss: 0.7109
Epoch 10/20 - Loss: 0.5598
Epoch 11/20 - Loss: 0.4254
Epoch 12/20 - Loss: 0.3219
Epoch 13/20 - Loss: 0.2471
Epoch 14/20 

In [22]:
bilstm_results_table = pd.DataFrame(
    bilstm_results
).T

bilstm_results_table

,Precision,Recall,F1
Barishal,0.819113,0.603015,0.694645
Chittagong,0.746269,0.512821,0.607903
Mymensingh,0.816393,0.630380,0.711429
Noakhali,0.735786,0.556962,0.634006
Sylhet,0.832685,0.536341,0.652439


In [23]:
def entity_diagnostics(model, test_loader):

    model.eval()

    gold_count = 0
    predicted_count = 0
    correct_count = 0

    with torch.no_grad():

        for tokens, tags, mask in test_loader:

            tokens = tokens.to(device)
            mask = mask.to(device)

            predictions = model(
                tokens,
                mask=mask
            )

            for i, pred in enumerate(predictions):

                length = mask[i].sum().item()

                true_tags = [
                    id_to_tag[x.item()]
                    for x in tags[i][:length]
                ]

                pred_tags = [
                    id_to_tag[x]
                    for x in pred
                ]

                true_entities = get_entities(true_tags)
                pred_entities = get_entities(pred_tags)

                gold_count += len(true_entities)
                predicted_count += len(pred_entities)
                correct_count += len(
                    true_entities & pred_entities
                )

    print("Gold entities:      ", gold_count)
    print("Predicted entities: ", predicted_count)
    print("Correct entities:   ", correct_count)

In [24]:
entity_diagnostics(
    bilstm_models["Noakhali"],
    test_loaders["Noakhali"]
)

Gold entities:       395
Predicted entities:  299
Correct entities:    220


In [25]:
print("Diagnostic:")
print("Precision:", 215 / 513)
print("Recall:", 215 / 395)

f1 = 2 * (215 / 513) * (215 / 395) / (
    (215 / 513) + (215 / 395)
)

print("F1:", f1)

Diagnostic:
Precision: 0.4191033138401559
Recall: 0.5443037974683544
F1: 0.473568281938326
